# Reddit Collection — PRAW (free tier)

Keyword-targeted collection of posts and comments from selected subreddits using the official
Python Reddit API Wrapper.

**Why PRAW:** the official API is the only sanctioned route to Reddit data, and its free tier was
sufficient. The tradeoff is rate limiting — collection ran across multiple sessions rather than in
one pass. The async variant in `02_reddit_scraper_async_praw.ipynb` was written to raise throughput
within the same limits.

**Output:** raw post/comment records for downstream filtering.


In [ ]:
pip install praw


In [ ]:
import pandas as pd
import praw
from praw.models import MoreComments

In [ ]:

reddit = praw.Reddit(user_agent="Comment Extraction (by /u/doko_mitendayo)",
                     client_id="xetbErZlqeoZdm5poeleqg", client_secret="XJJVw6k7Hf-HmVwSQxpmyV2lqFVGRQ")

In [ ]:
comments_data = []

In [ ]:
import pandas as pd

# Define keywords to search for
keywords = ['LGBTQ', 'pride', 'gay', 'trans', 'lesbian', 'LGBT']

# Subreddit to search within
subreddits = ['TrueUnpopularOpinion', 'Conservative', 'Politics']

In [ ]:


# Search in the specific subreddits
for subreddit_name in subreddits:
    subreddit = reddit.subreddit(subreddit_name)
    for submission in subreddit.search(' OR '.join(keywords), limit=100):
        submission.comments.replace_more(limit=0)
        for comment in submission.comments.list():
            comments_data.append([comment.body, submission.title])

# Create DataFrame to store data
comments_df = pd.DataFrame(comments_data, columns=['Comment', 'Post Title'])

# Save to CSV
comments_df.to_csv('lgbtq_comments.csv', index=False)


In [ ]:
comments_df.head()

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download the VADER lexicon for sentiment analysis
nltk.download('vader_lexicon')

# Initialize the Sentiment Intensity Analyzer
sia = SentimentIntensityAnalyzer()

# Function to apply VADER sentiment analysis to each comment
def get_sentiment_scores(comment):
    sentiment = sia.polarity_scores(comment)
    return pd.Series([sentiment['neg'], sentiment['neu'], sentiment['pos'], sentiment['compound']])

# Apply sentiment analysis to each comment and create new columns
comments_df[['neg', 'neu', 'pos', 'compound']] = comments_df['Comment'].apply(get_sentiment_scores)

# Create a new dataframe with the sentiment columns added
all_comments = comments_df

# Save the new dataframe to a CSV file
all_comments.to_csv('lgbtq_hate_comments_with_sentiment.csv', index=False)

# Display the first few rows of the new dataframe
print(all_comments.head())

# Analyze sentiment of comments
def is_hate_speech(comment):
    sentiment = sia.polarity_scores(comment)
    return sentiment['neg'] > 0.8  # Adjust the threshold for hate detection

# Filter hateful comments
hate_comments = comments_df[comments_df['Comment'].apply(is_hate_speech)]

# Save filtered hate comments to a CSV file
hate_comments.to_csv('filtered_hate_comments3.csv', index=False)


In [ ]:
# import nltk
# from nltk.sentiment import SentimentIntensityAnalyzer

# # Download the VADER lexicon for sentiment analysis
# nltk.download('vader_lexicon')

# # Initialize the Sentiment Intensity Analyzer
# sia = SentimentIntensityAnalyzer()

# # Analyze sentiment of comments
# def is_hate_speech(comment):
#     sentiment = sia.polarity_scores(comment)
#     return sentiment['neg'] > 0.8  # Adjust the threshold for hate detection

# # Filter hateful comments
# hate_comments = comments_df[comments_df['Comment'].apply(is_hate_speech)]

# # Save filtered hate comments to a CSV file
# hate_comments.to_csv('filtered_hate_comments3.csv', index=False)


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/lgbtq_hate_comments_with_sentiment.csv')

In [ ]:
df.head()

In [ ]:
import pandas as pd

# Assuming your DataFrame is named df
# Define a function to count the number of words in the 'Comment' column
def word_count(text):
    return len(str(text).split())

# Apply the function to the 'Comment' column and create a new column 'word_count'
df['word_count'] = df['Comment'].apply(word_count)

# Categorize the word count into specific bins: 1, 2, ..., 11, and 12+
bins = list(range(1, 13)) + [float('inf')]  # Bins for 1 to 12+ words
labels = [str(i) for i in range(1, 12)] + ['12+']  # Labels for each bin

# Create a column 'word_count_group' with the categorized word counts
df['word_count_group'] = pd.cut(df['word_count'], bins=bins, labels=labels, right=False)

# Count the number of occurrences for each word count group
word_count_distribution = df['word_count_group'].value_counts().sort_index()

# Print the result
print(word_count_distribution)


In [ ]:
def word_count(text):
    return len(str(text).split())

# Apply the function to the 'Comment' column and create a new column 'word_count'
df['word_count'] = df['Comment'].apply(word_count)

# Remove comments with less than 5 words
df_filtered = df.loc[df['word_count'] >= 5]

In [ ]:
df.to_csv("LT5_cleaned_comments.csv", index = False)

In [ ]:
import requests
import os

# Define keywords to search for
keywords = ['LGBTQ', 'pride', 'gay', 'trans', 'lesbian', 'LGBT']

# Subreddits to search within
subreddits = ['TrueUnpopularOpinion', 'Conservative', 'Politics']

# Create a directory to store images
if not os.path.exists('reddit_images'):
    os.makedirs('reddit_images')

# Function to check if any keyword exists in the title or text of the post
def contains_keyword(text, keywords):
    text_lower = text.lower()  # Make the text lowercase for case-insensitive matching
    return any(keyword.lower() in text_lower for keyword in keywords)

# Scrape the specified subreddits
for subreddit_name in subreddits:
    subreddit = reddit.subreddit(subreddit_name)
    print(f'Searching in subreddit: {subreddit_name}')

    for submission in subreddit.hot(limit=100):  # You can adjust the limit as needed
        # Check if the post title or text contains any of the keywords
        if contains_keyword(submission.title, keywords) or contains_keyword(submission.selftext, keywords):
            # Check if the post contains an image URL
            if submission.url.endswith(('jpg', 'jpeg', 'png', 'gif')):
                image_url = submission.url
                image_name = image_url.split('/')[-1]

                # Download the image
                try:
                    img_data = requests.get(image_url).content
                    with open(f'reddit_images/{image_name}', 'wb') as img_file:
                        img_file.write(img_data)
                        print(f'Downloaded {image_name} from {subreddit_name}')
                except Exception as e:
                    print(f'Failed to download {image_name}: {e}')

In [ ]:
import praw
import requests
import os

# Define keywords to search for
keywords = ['LGBTQ', 'pride', 'gay', 'trans', 'lesbian', 'LGBT']

# Subreddits to search within
subreddits = ['TrueUnpopularOpinion', 'Conservative', 'Politics']

# Create a directory to store images
if not os.path.exists('reddit_images'):
    os.makedirs('reddit_images')

# Function to check if any keyword exists in the title or text of the post
def contains_keyword(text, keywords):
    if text is None:
        return False
    text_lower = text.lower()  # Make the text lowercase for case-insensitive matching
    return any(keyword.lower() in text_lower for keyword in keywords)

# Function to download the image
def download_image(image_url, image_name):
    try:
        img_data = requests.get(image_url).content
        with open(f'reddit_images/{image_name}', 'wb') as img_file:
            img_file.write(img_data)
        print(f'Downloaded {image_name}')
    except Exception as e:
        print(f'Failed to download {image_name}: {e}')

# Initialize a counter for images encountered
image_count = 0

# Scrape the specified subreddits
for subreddit_name in subreddits:
    subreddit = reddit.subreddit(subreddit_name)
    print(f'Searching in subreddit: {subreddit_name}')

    for submission in subreddit.hot(limit=20000):  # Increase the limit for more posts
        print(f"Processing post: {submission.title} | URL: {submission.url}")  # Debugging info

        # Check if the post title or text contains any of the keywords
        if contains_keyword(submission.title, keywords) or contains_keyword(submission.selftext, keywords):
            # Check if the post contains an image URL (including Reddit-hosted and external links)
            if submission.url.endswith(('jpg', 'jpeg', 'png', 'gif')):
                image_url = submission.url
                image_name = image_url.split('/')[-1]
                download_image(image_url, image_name)
                image_count += 1  # Increment the counter when an image is found
            elif 'imgur.com' in submission.url or 'i.redd.it' in submission.url:
                # For Imgur or i.redd.it links, download the image
                image_url = submission.url
                if not image_url.endswith(('jpg', 'jpeg', 'png', 'gif')):
                    image_url += '.jpg'  # Assume it's a jpg if no extension is provided
                image_name = image_url.split('/')[-1]
                download_image(image_url, image_name)
                image_count += 1  # Increment the counter for Imgur or Reddit-hosted images

# After scraping, print the total number of images found
print(f"Total images encountered: {image_count}")
print('Scraping complete.')


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.tumblr.com/tagged/lgbt%20memes"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

for img_tag in soup.find_all('img'):
    img_url = img_tag['src']
    print(img_url)
